# Análise Diagnóstica — Onde e por que a margem deteriora

Este notebook foca em localizar os vetores de queda de margem (produto, desconto, envio, região e segmento) e identificar combinações críticas com alto impacto.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

def norm_ascii(s: str) -> str:
    s = '' if s is None else str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(s.split()).lower()

DATA_PATH = Path('..') / 'data' / 'raw' / 'Case 1 - Case Retail Store Sales Data.xlsx'
df = pd.read_excel(DATA_PATH, sheet_name='Sales_retail_store')

rename = {}
for c in df.columns:
    k = norm_ascii(c)
    if k == 'regiao':
        rename[c] = 'Região'
    if k == 'preco unitario':
        rename[c] = 'Preço Unitário'
if rename:
    df = df.rename(columns=rename)

df['Data da Venda'] = pd.to_datetime(df['Data da Venda'])
df['Ano'] = df['Data da Venda'].dt.year

for c in ['Faturamento', 'Lucro', 'Desconto', 'Custo de Envio']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

df['Faixa de Desconto'] = pd.cut(
    df['Desconto'],
    bins=[-1, 0, 0.05, 0.10, 0.20, 0.30, 1],
    labels=['0%', '0-5%', '5-10%', '10-20%', '20-30%', '30%+'],
)

df.shape

## Comparativo 2025 → 2026 por recorte
Para cada recorte, observamos variação de faturamento, variação de lucro e mudança de margem (em p.p.).

In [ ]:
def agg(df_, by):
    g = (
        df_.groupby(by)
           .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'), pedidos=('Order ID','nunique'))
           .reset_index()
    )
    g['margem'] = np.where(g['faturamento'] != 0, g['lucro'] / g['faturamento'], np.nan)
    return g

def delta_2526(by):
    a = agg(df[df['Ano']==2025], by).rename(columns={'faturamento':'fat_2025','lucro':'luc_2025','margem':'marg_2025'})
    b = agg(df[df['Ano']==2026], by).rename(columns={'faturamento':'fat_2026','lucro':'luc_2026','margem':'marg_2026'})
    out = a.merge(b, on=by, how='outer').fillna(0)
    out['d_fat'] = out['fat_2026'] - out['fat_2025']
    out['d_luc'] = out['luc_2026'] - out['luc_2025']
    out['d_marg_pp'] = (out['marg_2026'] - out['marg_2025']) * 100
    return out

delta_2526(['Categoria do Produto']).sort_values('d_luc').head(10)

In [ ]:
delta_2526(['Categoria do Produto','Sub-Categoria do Produto']).sort_values('d_luc').head(15)

In [ ]:
delta_2526(['Forma de Envio']).sort_values('d_luc')

In [ ]:
delta_2526(['Região']).sort_values('d_luc')

In [ ]:
delta_2526(['Segmento do Cliente']).sort_values('d_luc')

## Combinações críticas (produto × envio)
Aqui priorizamos combinações com margem baixa/negativa e relevância (participação no faturamento de 2026).

In [ ]:
base26 = df[df['Ano']==2026].copy()
fat26 = base26['Faturamento'].sum() or 1

combo = (
    base26.groupby(['Categoria do Produto','Sub-Categoria do Produto','Forma de Envio'], as_index=False)
          .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
combo['margem'] = np.where(combo['faturamento'] != 0, combo['lucro']/combo['faturamento'], np.nan)
combo['share_2026'] = combo['faturamento'] / fat26

combo[combo['share_2026'] >= 0.01].sort_values('lucro').head(20)

## Faixas de desconto e erosão de margem
Este corte ajuda a diferenciar queda de margem causada por desconto versus logística (envio).

In [ ]:
tech26 = base26[base26['Categoria do Produto']=='Tecnologia']
disc = (
    tech26.groupby('Faixa de Desconto', as_index=False)
          .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
disc['margem'] = np.where(disc['faturamento'] != 0, disc['lucro']/disc['faturamento'], np.nan)
disc.sort_values('faturamento', ascending=False)